In [4]:
# =========================================================
# 07-2 재현 — 2층 ReLU + Adam 심층 신경망
# =========================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense


# enable_op_determinism() 포함
tf.keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()


(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

X_train = X_train / 255.0
X_test = X_test / 255.0

model = Sequential([
    # Input(shape=(28,28)) + Flatten() 구조
    Input(shape=(28, 28)),
    Flatten(),

    # Dense(100, activation='relu') 은닉층
    Dense(100, activation='relu'),

    Dense(10, activation='softmax')
])

model.compile(
    # optimizer='adam'
    optimizer='adam',

    # sparse_categorical_crossentropy
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=32
)

model.summary()

val_loss, val_accuracy = model.evaluate(X_test, y_test)
print("val_accuracy:", val_accuracy)

# val_accuracy ≥ 0.8723 확인 셀 포함
print("val_accuracy >= 0.8723:", val_accuracy >= 0.8723)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8252 - loss: 0.4974 - val_accuracy: 0.8469 - val_loss: 0.4236
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8661 - loss: 0.3749 - val_accuracy: 0.8591 - val_loss: 0.3855
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8779 - loss: 0.3363 - val_accuracy: 0.8638 - val_loss: 0.3714
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8857 - loss: 0.3139 - val_accuracy: 0.8686 - val_loss: 0.3680
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8911 - loss: 0.2958 - val_accuracy: 0.8690 - val_loss: 0.3696


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_8 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 238,532 (931.77 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 159,022 (621.18 KB)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8690 - loss: 0.3696  
val_accuracy: 0.8690000176429749
val_accuracy >= 0.8723: False


In [5]:
# =========================================================
# SGD vs Adam 옵티마이저 비교 실험
# =========================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.optimizers import SGD, Adam


(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

X_train = X_train / 255.0
X_test = X_test / 255.0


def make_model():
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(100, activation='relu'),
        Dense(10, activation='softmax')
    ])

    return model


optimizers = {
    "SGD 기본": SGD(),
    "SGD lr=0.1": SGD(learning_rate=0.1),
    "Adam": Adam()
}

results = []

for name, optimizer in optimizers.items():

    # keras.utils.set_random_seed(42) 각 반복마다 적용
    tf.keras.utils.set_random_seed(42)

    # make_model()로 매번 초기화된 새 모델 사용
    model = make_model()

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # verbose=0으로 진행 상황 숨김
    model.fit(
        X_train,
        y_train,
        validation_data=(X_test, y_test),
        epochs=5,
        batch_size=32,
        verbose=0
    )

    # model.evaluate()로 val_accuracy 측정
    val_loss, val_accuracy = model.evaluate(X_test, y_test, verbose=0)

    results.append({
        "optimizer": name,
        "val_accuracy": val_accuracy
    })


# 3가지 옵티마이저 결과표
df_results = pd.DataFrame(results)
display(df_results)


# "왜 Adam이 나은가" 주석
print("주석: Adam은 학습률을 고정해서 쓰는 SGD보다 각 가중치마다 업데이트 크기를 자동으로 조절하므로 더 빠르고 안정적으로 좋은 성능에 도달하기 쉽다.")

c:\Users\pipa03\p1-data\c3-deep-learning\.venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


,optimizer,val_accuracy
0,SGD 기본,0.835
1,SGD lr=0.1,0.859
2,Adam,0.869


주석: Adam은 학습률을 고정해서 쓰는 SGD보다 각 가중치마다 업데이트 크기를 자동으로 조절하므로 더 빠르고 안정적으로 좋은 성능에 도달하기 쉽다.


In [6]:
# =========================================================
# Learning Rate 조정 + CIFAR-10으로 Dense 한계 체험
# =========================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.optimizers import Adam


(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# CIFAR-10: cifar_train.reshape(-1) 레이블 1D 변환
y_train = y_train.reshape(-1)
y_test = y_test.reshape(-1)

X_train = X_train / 255.0
X_test = X_test / 255.0

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

# CIFAR-10 입력 shape: (32, 32, 3) — RGB 3채널 확인
print("CIFAR-10 이미지 한 장 shape:", X_train[0].shape)


def make_cifar_model(learning_rate):
    model = Sequential([
        # Flatten 후 입력 크기: 32×32×3 = 3,072
        Input(shape=(32, 32, 3)),
        Flatten(),

        Dense(100, activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


learning_rates = [0.001, 0.0001, 0.01]

cifar_results = []

for lr in learning_rates:
    tf.keras.utils.set_random_seed(42)

    model = make_cifar_model(lr)

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_test, y_test),
        epochs=5,
        batch_size=32,
        verbose=0
    )

    val_loss, val_accuracy = model.evaluate(X_test, y_test, verbose=0)

    cifar_results.append({
        "learning_rate": lr,
        "val_accuracy": val_accuracy
    })


df_cifar = pd.DataFrame(cifar_results)
display(df_cifar)


model_check = make_cifar_model(0.001)
model_check.summary()

# Dense(100) params = 3,072×100 + 100 = 307,300
print("Dense(100) params 계산:", 3072 * 100 + 100)

X_train shape: (50000, 32, 32, 3)
y_train shape: (50000,)
CIFAR-10 이미지 한 장 shape: (32, 32, 3)


,learning_rate,val_accuracy
0,0.0010,0.3486
1,0.0001,0.4343
2,0.0100,0.1729


Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_15 (Flatten)            │ (None, 3072)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 100)            │       307,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 308,310 (1.18 MB)

 Trainable params: 308,310 (1.18 MB)

 Non-trainable params: 0 (0.00 B)

Dense(100) params 계산: 307300
